# 04 — Build a baseline and make classification metrics concrete

**Plain-language question:** Can a model look accurate while finding no
churners?

**Why this matters:** “85% accurate” sounds impressive until you compare it
with a rule that always predicts the common class.

**Estimated time:** 50–60 minutes.
**Prerequisite:** lessons 00–03; you know the positive class and the roles of
train and validation.


## Preflight

Run this check, then load only train and validation.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, confusion_matrix

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.workflow import ensure_prepared, run_baseline

ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
y_validation = validation.churned_30d
print(f"✓ Validation rows: {len(validation):,}")


### What you should see

360 validation rows. It is legitimate to inspect validation labels because
validation's declared job is model development. Test labels remain sealed.

### Words introduced

| Word | Plain meaning | Question answered |
|---|---|---|
| baseline | A deliberately simple comparator | Do we beat no skill? |
| confusion matrix | Counts of four correct/error outcomes | What mistakes occur? |
| recall | Share of true positives that we found | How many churners were caught? |


## Start without scikit-learn: predict every row as no churn

**Before you run this:** because churn is the uncommon class, predict whether
accuracy will be high or low—and what recall will be.


In [ ]:
all_negative = np.zeros(len(y_validation), dtype=int)
tn, fp, fn, tp = confusion_matrix(y_validation, all_negative, labels=[0, 1]).ravel()

pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["actual 0", "actual 1"],
    columns=["predicted 0", "predicted 1"],
)


### What you should see

290 true negatives, 0 false positives, 70 false negatives, and 0 true
positives. The rule is correct on all 290 non-churn rows and misses all 70 churn
rows.


## Calculate accuracy, precision, and recall from those counts

- **Accuracy** = all correct predictions / all rows.
- **Precision** = true positives / all predicted positives.
- **Recall** = true positives / all actual positives.

When there are no predicted positives, this course reports precision as zero.


In [ ]:
baseline_arithmetic = pd.Series(
    {
        "accuracy": (tn + tp) / (tn + fp + fn + tp),
        "precision": 0.0 if tp + fp == 0 else tp / (tp + fp),
        "recall": tp / (tp + fn),
    },
    name="value",
)
baseline_arithmetic.to_frame()


### How to interpret the output

Accuracy is about 80.6%, but recall and precision are 0%. Accuracy is not
mathematically wrong; by itself it answers the wrong practical question for an
uncommon positive event.

### Misconception check

A high accuracy percentage does not imply that the model finds any churners.
Always inspect the error counts and metrics connected to the intended action.


## A probability baseline and a ranking metric

`DummyClassifier(strategy="prior")` assigns every row the training churn rate.
At threshold 0.5, every prediction is still negative. **Average precision (AP)**
asks whether positive rows tend to receive higher scores than negative rows; a
constant-score baseline has AP near the positive prevalence.


In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="prior")
dummy.fit(np.zeros((len(train), 1)), train.churned_30d)
dummy_scores = dummy.predict_proba(np.zeros((len(validation), 1)))[:, 1]

pd.Series(
    {
        "constant_score": dummy_scores[0],
        "validation_prevalence": y_validation.mean(),
        "average_precision": average_precision_score(y_validation, dummy_scores),
    }
).to_frame("value")


### What you should see

One constant training-prior score (about 0.151), validation prevalence about
0.194, and average precision about 0.194. The baseline cannot rank one
validation account above another.

Now record the same concept as a repeatable MLflow baseline run.


In [ ]:
baseline_evidence = run_baseline(settings, root)
baseline_metrics = baseline_evidence["metrics"]
pd.Series(
    {
        "run_id": baseline_evidence["run_id"],
        "validation_accuracy": baseline_metrics["accuracy"],
        "validation_recall": baseline_metrics["recall"],
        "validation_average_precision": baseline_metrics["average_precision"],
    }
).to_frame("recorded value")


### MLflow words introduced

| Word | Plain meaning | Baseline example |
|---|---|---|
| experiment | A collection of related attempts | subscription-churn course |
| run | One recorded execution | this baseline fit/evaluation |
| parameter / metric / artifact | input choice / measured number / saved file | strategy / recall / model |

The long run ID is a lookup key, not a result to interpret. MLflow also stores
the dataset fingerprint, model artifact, dependency evidence, and source state.
Lesson 05 will make the trained pipeline itself visible.


### Guided exercise

Predict every validation account as churn. Compute the four confusion counts
and teaching cost. A false negative costs 5; a false positive costs 1.


In [ ]:
exercise_all_positive = np.ones(len(y_validation), dtype=int)
exercise_tn, exercise_fp, exercise_fn, exercise_tp = confusion_matrix(
    y_validation, exercise_all_positive, labels=[0, 1]
).ravel()
exercise_cost = (
    exercise_fn * settings.selection.false_negative_cost
    + exercise_fp * settings.selection.false_positive_cost
)
pd.Series(
    {
        "tn": exercise_tn,
        "fp": exercise_fp,
        "fn": exercise_fn,
        "tp": exercise_tp,
        "cost": exercise_cost,
    }
)


**Self-check:** all-positive should find every churner but create many
unnecessary reviews.

<details><summary>Solution explanation</summary>

The counts are TN=0, FP=290, FN=0, TP=70. Recall is 100%, but precision and
accuracy are only 19.4%. Its cost is 290, compared with 350 for all-negative
under the fictional weights. Neither rule ranks accounts or balances capacity.
</details>


In [ ]:
# Reference solution — run after your attempt
assert (exercise_tn, exercise_fp, exercise_fn, exercise_tp) == (0, 290, 0, 70)
assert exercise_cost == 290
print("✓ All-positive trades perfect recall for 290 false positives")


## MLOps bridge

A candidate should never be called an improvement without a baseline recorded
on the same validation population. MLflow makes that comparator discoverable
and binds its metrics to exact data and code evidence.

## Recap

- A confusion matrix exposes the exact correct and incorrect outcomes.
- The all-negative baseline looks accurate but has zero recall.
- Average precision evaluates ranking; the no-skill value is near prevalence.

**Evidence created:** one reusable MLflow baseline run plus `baseline.json` in
course state. Rerunning replaces no selection or test evidence; it records a
fresh baseline result on the same verified inputs.

**Ready for 05?** You can calculate accuracy and recall from TN/FP/FN/TP and
explain why the baseline is not useful.
